# Wrangling Tasks

## Convert formats (CSV → structured tables)

In [1]:
# import libraries first
import pandas as pd
import numpy as np 

### Tree canopies 2021 urban forest data

In [2]:
# 1. Shape and basic structure
Tree_canopy_2021 = pd.read_csv("tree-canopies-2021-urban-forest.csv")
print(Tree_canopy_2021.shape)
Tree_canopy_2021.head()

(57980, 2)


,Geo Point,Geo Shape
0,"-37.795401772079884, 144.9578491713787","{""coordinates"": [[[[144.95785054828673, -37.79..."
1,"-37.79535552368289, 144.9514339834316","{""coordinates"": [[[[144.95143147647892, -37.79..."
2,"-37.79569214657717, 144.97057737168868","{""coordinates"": [[[[144.9705639020907, -37.795..."
3,"-37.794822540469625, 144.9232541798998","{""coordinates"": [[[[144.9233177524972, -37.794..."
4,"-37.79530213879436, 144.9476228687511","{""coordinates"": [[[[144.94762531525055, -37.79..."


The 2021 tree canopy dataset contains 57,980 rows and two columns, where Geo Point, a single lat/long coordinate marking each canopy polygon's centroid, and Geo Shape, a GeoJSON string describing the polygon's full boundary is present. This means each row is one distinct patch of tree canopy, not one individual tree.

In [3]:
# 2. Data types to find which columns to expect:
Tree_canopy_2021.dtypes

Geo Point    object
Geo Shape    object
dtype: object

In [4]:
# 3. Missing values: how many nulls per column?
Tree_canopy_2021.isnull().sum()

Geo Point    0
Geo Shape    0
dtype: int64

In [5]:
# 4. Duplicates in the rows?
Tree_canopy_2021.duplicated().sum()

np.int64(0)

In [6]:
# 5. Summary statistics for numeric columns
Tree_canopy_2021.describe()

,Geo Point,Geo Shape
count,57980,57980
unique,57980,57980
top,"-37.795401772079884, 144.9578491713787","{""coordinates"": [[[[144.95785054828673, -37.79..."
freq,1,1


Since Geo Point and Geo Shape are both text columns rather than numeric ones, .describe() falls back to count, unique, top, and frequency instead of numeric summary statistics. The output shows count and unique are both 57,980 for each column, meaning every row has a distinct coordinate and a distinct polygon shape, and the most frequent value in each column only appears once (freq = 1). This confirms there are no duplicate canopy records in the dataset. Each of the 57,980 rows represents a genuinely separate patch of tree canopy.

In [7]:
# 6. Unique values for categorical/text columns
Tree_canopy_2021['Geo Point'].nunique()  # should roughly match row count if each is unique

57980

In [8]:
# 7. Column info all at once (dtype + non-null counts)
Tree_canopy_2021.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57980 entries, 0 to 57979
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Geo Point  57980 non-null  object
 1   Geo Shape  57980 non-null  object
dtypes: object(2)
memory usage: 906.1+ KB


There are 57,980 non-null values out of 57,980 rows in each column, so no missing data anywhere. Both are stored as object dtype (pandas' generic type for text/strings), which makes sense since Geo Point is a coordinate string and Geo Shape is a JSON string, not actual numbers pandas can do math on yet. The dataset takes up about 906 KB in memory, small enough to work with directly.

In [9]:
# Check Geo Point format consistency
Tree_canopy_2021['Geo Point'].str.split(',').apply(len).value_counts()
# should all be 2 (lat, lon) — flags any malformed rows

Geo Point
2    57980
Name: count, dtype: int64

In [10]:
# Check for nulls in either column
Tree_canopy_2021.isnull().sum()

Geo Point    0
Geo Shape    0
dtype: int64

In [11]:
# Check for nulls in either column
Tree_canopy_2021.isnull().sum()

Geo Point    0
Geo Shape    0
dtype: int64

In [12]:
# Peek at a full Geo Shape value (not truncated) to understand its structure
print(Tree_canopy_2021['Geo Shape'].iloc[0])

{"coordinates": [[[[144.95785054828673, -37.79533581409328], [144.95786144538243, -37.7953360030544], [144.9578613858508, -37.79533816484839], [144.95786411012483, -37.79533821208852], [144.95786393153014, -37.79534469747047], [144.9578666558044, -37.79534474471054], [144.95786641767833, -37.795353391886465], [144.95787731477674, -37.795353580846204], [144.9578771957143, -37.795357904434184], [144.9578826442639, -37.79535799891369], [144.95788270379492, -37.795355837119686], [144.95788542806966, -37.795355884359346], [144.95788536853865, -37.795358046153346], [144.95789354136306, -37.79535818787194], [144.95789348183231, -37.79536034966596], [144.9579016557921, -37.795360491403684], [144.9579015962616, -37.7953626531977], [144.9579043205366, -37.79536270043694], [144.95790426100615, -37.79536486223096], [144.95791515810652, -37.79536505118725], [144.95791509857636, -37.79536721298128], [144.95791782285156, -37.795367260220196], [144.9579177633215, -37.79536942201425], [144.957920487596

Inspecting a full Geo Shape value confirms each canopy polygon is stored as a GeoJSON MultiPolygon string, with nested coordinate arrays tracing the canopy boundary. 

### Trees, with species and dimensions (Urban Forest)

#### Dataset inspections

In [13]:
Trees = pd.read_csv("trees-with-species-and-dimensions-urban-forest.csv")
print(Trees.shape)
Trees.head()

(82064, 20)


,com_id,common_name,scientific_name,genus,family,diameter_breast_height,year_planted,date_planted,age_description,useful_life_expectency,useful_life_expectency_value,precinct,located_in,uploaddate,coordinatelocation,latitude,longitude,easting,northing,geolocation
0,1023218,London Plane,Platanus x acerifolia,Platanus,Platanaceae,27.0,1900,1900-01-01,Mature,31 - 40 years,40,Carlton,Street,NaN,"-37.80498629, 144.968114",-37.804986,144.968114,321125.06,5813877.14,"-37.80498629, 144.968114"
1,1023221,London Plane,Platanus x acerifolia,Platanus,Platanaceae,35.0,1900,1900-01-01,Mature,31 - 40 years,40,Carlton,Street,NaN,"-37.80502242, 144.96843217",-37.805022,144.968432,321153.16,5813873.74,"-37.80502242, 144.96843217"
2,1023222,London Plane,Platanus x acerifolia,Platanus,Platanaceae,45.0,1900,1900-01-01,Mature,21 - 30 years,30,Carlton,Street,NaN,"-37.80505925, 144.96873896",-37.805059,144.968739,321180.25,5813870.24,"-37.80505925, 144.96873896"
3,1023224,London Plane,Platanus x acerifolia,Platanus,Platanaceae,NaN,1900,1900-01-01,Mature,31 - 40 years,40,Carlton,Street,NaN,"-37.80524724, 144.9683783",-37.805247,144.968378,321148.95,5813848.69,"-37.80524724, 144.9683783"
4,1023225,London Plane,Platanus x acerifolia,Platanus,Platanaceae,NaN,1900,1900-01-01,Mature,NaN,50,Carlton,Street,NaN,"-37.80521592, 144.96807817",-37.805216,144.968078,321122.45,5813851.59,"-37.80521592, 144.96807817"


The trees dataset contains 82,064 rows, one per individually managed tree — unlike the canopy dataset, which is one row per canopy polygon. Each row carries species identity (common_name, scientific_name, genus, family), physical and lifecycle attributes (diameter_breast_height, year_planted, age_description, useful_life_expectency), administrative location (precinct, located_in), and the tree's coordinates, duplicated across four different formats (coordinatelocation, latitude/longitude, easting/northing, geolocation). Only latitude/longitude is needed for the spatial join against the canopy polygons.

In [14]:
Trees.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82064 entries, 0 to 82063
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   com_id                        82064 non-null  int64  
 1   common_name                   82058 non-null  object 
 2   scientific_name               82064 non-null  object 
 3   genus                         82064 non-null  object 
 4   family                        82064 non-null  object 
 5   diameter_breast_height        37201 non-null  float64
 6   year_planted                  82064 non-null  int64  
 7   date_planted                  82064 non-null  object 
 8   age_description               82064 non-null  object 
 9   useful_life_expectency        64654 non-null  object 
 10  useful_life_expectency_value  82064 non-null  int64  
 11  precinct                      82064 non-null  object 
 12  located_in                    82064 non-null  object 
 13  u

The `.info()` shows `uploaddate` is entirely empty (0/82,064 non-null), so it can be dropped outright. The `common_name` is missing for only 6 of 82,064 rows. The `useful_life_expectency` is missing for 17,410 rows (about 21%), which is too large a share to drop, so these should be labelled 'Unknown' instead of left null, preserving the row while flagging the missing value explicitly. The `diameter_breast_height` is missing for over half the rows (44,863/82,064), so I want to drop it as well. Every other column is fully populated and needs no cleaning.

In [15]:
Trees.isnull().sum()

com_id                              0
common_name                         6
scientific_name                     0
genus                               0
family                              0
diameter_breast_height          44863
year_planted                        0
date_planted                        0
age_description                     0
useful_life_expectency          17410
useful_life_expectency_value        0
precinct                            0
located_in                          0
uploaddate                      82064
coordinatelocation                  0
latitude                            0
longitude                           0
easting                             0
northing                            0
geolocation                         0
dtype: int64

The `uploaddate` is fully null, `diameter_breast_height` and ``useful_life_expecteny` have big gaps (44,863 and 17,410 rows), `common_name` is missing 6 rows, and all other columns are complete.

In [16]:
# To find out the most common species of Tree:
Trees['common_name'].value_counts().head(20)

common_name
River red gum          8338
London Plane           5070
Drooping sheoak        3452
English Elm            3291
Yellow Box             3043
Black Wattle           3001
Spotted Gum            2902
River Sheoak           1665
Lightwood Wattle       1619
Sweet Bursaria         1574
Lemon Scented Gum      1402
Smooth-barked apple    1326
Yellow Gum             1235
Golden Wattle          1035
Chinaberry Tree         971
Brush Box               895
Coastal Banksia         786
Silver Wattle           771
Dutch Elm               740
Moreton Bay Fig         711
Name: count, dtype: int64

The `value_counts()` on `common_name` shows River red gum (8,338), London Plane (5,070), and Drooping sheoak (3,452) as the three most planted species, with counts dropping off steadily after that. This confirms a small number of species make up a large share of the dataset, should make another category by genus down to just the top 8 rather than trying to show every distinct species.

In [17]:
# To see what kind of precints are reported here 
Trees['precinct'].value_counts()

precinct
Parkville                   28567
Melbourne                    9063
Kensington                   7555
West Melbourne               6756
Docklands                    5816
North Melbourne              4715
East Melbourne               4506
Carlton                      4368
South Yarra                  2631
Port Melbourne               2623
Southbank                    2338
Carlton North                2323
Flemington                    510
North And West Melbourne      113
Princes Hill                   65
Fishermans Bend                61
Central City                   26
Brunswick West                 24
Brunswick                       2
Richmond                        2
Name: count, dtype: int64

The `value_counts()` on precinct shows Parkville (28,567) dominates the dataset, followed by Melbourne (9,063), Kensington (7,555), and West Melbourne (6,756), with several precincts having very small counts (Brunswick and Richmond at 2 rows each). This confirms precinct is usable as-is, with 20 distinct values and no obvious inconsistencies that would need cleaning before it's used for the zone precinct-labelling step later.

In [18]:
# To be able to sanity check the coordinates:
Trees[['latitude','longitude']].describe()

,latitude,longitude
count,82064.000000,82064.000000
mean,-37.801873,144.950772
std,0.016644,0.019349
min,-37.850518,144.900370
25%,-37.815692,144.941931
50%,-37.797766,144.951071
75%,-37.789021,144.962702
max,-37.775511,144.991056


The summary statistics on latitude and longitude confirm all 82,064 rows have valid coordinates within the expected bounding box for the City of Melbourne (latitude -37.85 to -37.78, longitude 144.90 to 144.99), with no sign errors, swapped columns, or out-of-range outliers. This confirms the coordinates are safe to use directly in the spatial join against the canopy polygons.

### Spatial Join: Matching Trees to Canopy Polygons

References:
Shapely (general library): https://shapely.readthedocs.io/en/stable/
STRtree specifically: https://shapely.readthedocs.io/en/stable/strtree.html

To determine which canopy polygon each tree belongs to, the canopy dataset is first given a unique `polygon_id`, and each `Geo Shape` string is parsed into a real polygon geometry using Shapely.

The `!pip install` shapely installs the geometry library this whole spatial join depends on, and `from shapely.geometry import shape, Point / from shapely.strtree import STRtree` bring in the three tools actually used. The tools used below include `shape()` to turn parsed GeoJSON into real polygon objects, `Point` to represent each tree's coordinate as a queryable point, and `STRtree` to build the spatial index. An STRtree works like a search shortcut where it pre-organises all the polygons by their location, so that later, when you ask "which polygons are near this point?", it can jump straight to the small handful of polygons that could plausibly be nearby, instead of checking all 57,980 every time.

In [19]:
!pip install shapely

In [20]:
import json
from shapely.geometry import shape, Point
from shapely.strtree import STRtree

The canopy dataset has no ID column, only Geo Point and Geo Shape, so a `polygon_id` is created here to give each polygon something stable to be referenced by once trees are matched to it.

In [21]:
# Step 1: Give the canopy polygons a proper ID
# The 2021 canopy dataset has no ID column, so we create one
Tree_canopy_2021 = Tree_canopy_2021.reset_index(drop=True)
Tree_canopy_2021['polygon_id'] = Tree_canopy_2021.index  # simple integer ID, 0 to 57979

The `Geo Shape` is stored as plain text, so each value is parsed with `json.loads()` and converted into an actual Shapely polygon with `shape()`, giving geometry objects that can be queried spatially rather than just read as strings. The matching `polygon_id` is kept alongside each parsed polygon so the two lists stay aligned by position, and any row that fails to parse is skipped rather than crashing the loop.

In [22]:
# Step 2: Parse polygons into real geometry objects
polygons = []
polygon_ids = []
for idx, gs in Tree_canopy_2021['Geo Shape'].items():
    try:
        polygons.append(shape(json.loads(gs)))
        polygon_ids.append(Tree_canopy_2021.loc[idx, 'polygon_id'])
    except Exception:
        pass

print(f"Parsed {len(polygons)} polygons")

Parsed 57980 polygons


An STRtree is like a search shortcut where it pre-organises all the polygons by their location, so that later, when you ask "which polygons are near this point?", it can jump straight to the small handful of polygons that could plausibly be nearby, instead of checking all 57,980 every time.

In [23]:
# Step 3: Build spatial index for fast lookup 
tree_index = STRtree(polygons)

1. Turn the tree into a point.
The tree's lat/long becomes a Shapely Point object. This is something that can be checked against the polygons, not just two numbers.

2. First try: is the tree exactly inside a polygon?
The `pt.buffer(buffer_exact)` draws a tiny circle (radius  around 22m, since 0.0002 degrees) around the tree's point, and the STRtree returns only the handful of polygons that overlap that circle. This is the "search shortcut" from before, avoiding a check against all 57,980. Of those few candidates, it then asks each one directly: "do you actually contain this point?" If yes, a real, exact match — return immediately with `distance_m = 0.0`.

3. If nothing contained it: look wider for the nearest polygon.
No polygon actually contains the tree — but it might just be a few metres off due to GPS/survey imprecision. So the search radius widens (buffer_near, ~55m), grab every polygon in that wider circle, measure the exact distance from the tree to each one, and pick the closest. That distance (in degrees) gets converted to roughly metres by multiplying by 111,000 (the rough length of one degree of latitude at Melbourne). This is a 'nearby' match, and it's how you got the "median 2.85m" numbers from your validation step

4. If still nothing found even in the wider circle: no match.

In [24]:
# Step 4: For each tree, find its matching polygon 
def find_match(row, buffer_exact=0.0002, buffer_near=0.0005):
    pt = Point(row['longitude'], row['latitude'])
    
    # check exact containment first
    candidate_idxs = tree_index.query(pt.buffer(buffer_exact))
    for i in candidate_idxs:
        if polygons[i].contains(pt):
            return pd.Series({
                'com_id': row['com_id'],
                'polygon_id': polygon_ids[i],
                'match_type': 'exact',
                'distance_m': 0.0
            })
    
    # if no exact match, find nearest polygon within buffer
    nearby_idxs = tree_index.query(pt.buffer(buffer_near))
    if len(nearby_idxs) > 0:
        # find the closest one among candidates
        distances = [(i, pt.distance(polygons[i])) for i in nearby_idxs]
        nearest_i, nearest_dist = min(distances, key=lambda x: x[1])
        # convert degrees to approx metres (rough, at Melbourne's latitude)
        distance_m = nearest_dist * 111000
        return pd.Series({
            'com_id': row['com_id'],
            'polygon_id': polygon_ids[nearest_i],
            'match_type': 'nearby',
            'distance_m': round(distance_m, 1)
        })
    
    # no match at all
    return pd.Series({
        'com_id': row['com_id'],
        'polygon_id': None,
        'match_type': 'no_match',
        'distance_m': None
    })

For each tree, the function checks "am I exactly inside a polygon?" first, falls back to "what's the closest polygon nearby?" if not, and gives up with `no_match` only if nothing is found even in the wider search, with three tiers of increasing tolerance, which is exactly what the earlier `match_type` value counts (exact/nearby/no_match) are reporting on.

The `find_match()` is first tested on a random 300-tree sample (random_state=1 fixes the sample so it's reproducible), rather than run over the full 82,064 rows immediately, since matching is slow to run and a bug is far cheaper to catch on 300 rows than after committing to the full dataset. the `.apply(find_match, axis=1)` runs the function once per tree, returning `TREE_CANOPY_MATCH` with each tree's `com_id`, `matched polygon_id`, `match_type`, and `distance_m. .head()` and `.value_counts()` are a quick sanity check that the output looks correct and the match rates are reasonable before scaling up to the full dataset in the next step.

In [25]:
# Step 5: Run it (start with a sample to test, then scale up) 
sample = Trees.sample(300, random_state=1)
TREE_CANOPY_MATCH = sample.apply(find_match, axis=1)

print(TREE_CANOPY_MATCH.head())
print(TREE_CANOPY_MATCH['match_type'].value_counts())

        com_id  polygon_id match_type  distance_m
81477  1046970       19010      exact         0.0
51005  1048053       16283      exact         0.0
36517  1831433        4635     nearby        44.1
77798  1358650        5794      exact         0.0
69280  1029621        5144      exact         0.0
match_type
exact     192
nearby    108
Name: count, dtype: int64


All 57,980 canopy polygons parsed successfully. Of the 300-tree sample, 192 trees (64%) matched exactly inside a polygon and 108 (36%) matched to a nearby polygon instead, with no unmatched trees.

In [26]:
# How far away are the 'nearby' matches, on average?
TREE_CANOPY_MATCH[TREE_CANOPY_MATCH['match_type'] == 'nearby']['distance_m'].describe()

count    108.000000
mean       4.715741
std        5.730068
min        0.000000
25%        1.000000
50%        2.850000
75%        6.650000
max       44.100000
Name: distance_m, dtype: float64

Of the sampled trees not falling exactly within a mapped canopy polygon, the median distance to the nearest polygon was 2.85m and 75% were within 6.65m, consistent with expected GPS and survey precision differences between the per-tree and canopy datasets, rather than a genuine spatial inconsistency.

In [27]:
# Any trees with no match at all?
TREE_CANOPY_MATCH['match_type'].value_counts(normalize=True) * 100

match_type
exact     64.0
nearby    36.0
Name: proportion, dtype: float64

Spatial validation of a 300-tree random sample confirmed strong alignment between the two datasets: 64% of trees fell exactly within a mapped 2021 canopy polygon, and the remaining 36% were located a median of 2.85m from the nearest polygon, well within expected GPS and survey precision. No trees (0%) were unmatched to any canopy feature, confirming the two datasets are spatially consistent and suitable for combined use

In [28]:
!pip install shapely scikit-learn

### Cleaning the Trees Dataset

In [29]:
# Drop the fully-empty uploaddate column
Trees = Trees.drop(columns=['uploaddate'])

# Drop the 6 rows missing common_name (negligible loss)
Trees = Trees.dropna(subset=['common_name'])

# Label missing useful_life_expectency explicitly rather than leaving null
Trees['useful_life_expectency'] = Trees['useful_life_expectency'].fillna('Unknown')

print("Cleaned Trees shape:", Trees.shape)

Cleaned Trees shape: (82058, 19)


### Spatial join on full 82,064 rows + save CSV

In [30]:
def find_match(row, buffer_exact=0.0002, buffer_near=0.0005):
    pt = Point(row['longitude'], row['latitude'])
    
    candidate_idxs = tree_index.query(pt.buffer(buffer_exact))
    for i in candidate_idxs:
        if polygons[i].contains(pt):
            return pd.Series({
                'com_id': row['com_id'],
                'polygon_id': polygon_ids[i],
                'match_type': 'exact',
                'distance_m': 0.0
            })
    
    nearby_idxs = tree_index.query(pt.buffer(buffer_near))
    if len(nearby_idxs) > 0:
        distances = [(i, pt.distance(polygons[i])) for i in nearby_idxs]
        nearest_i, nearest_dist = min(distances, key=lambda x: x[1])
        distance_m = nearest_dist * 111000
        return pd.Series({
            'com_id': row['com_id'],
            'polygon_id': polygon_ids[nearest_i],
            'match_type': 'nearby',
            'distance_m': round(distance_m, 1)
        })
    
    return pd.Series({
        'com_id': row['com_id'],
        'polygon_id': None,
        'match_type': 'no_match',
        'distance_m': None
    })

# Run on full dataset (82,064 rows)
TREE_CANOPY_MATCH = Trees.apply(find_match, axis=1)

print(TREE_CANOPY_MATCH['match_type'].value_counts())
print(TREE_CANOPY_MATCH['match_type'].value_counts(normalize=True) * 100)

# Save so you don't have to recompute
TREE_CANOPY_MATCH.to_csv("tree_canopy_match.csv", index=False)

match_type
exact       50358
nearby      31685
no_match       15
Name: count, dtype: int64
match_type
exact       61.368788
nearby      38.612932
no_match     0.018280
Name: proportion, dtype: float64


### Building the Map Dataset to display Genus cateogries 

The `map_trees` selects the core attributes needed for the frontend map from Trees, then buckets genus into the top 8 most common genera plus an `'Other'`, giving a manageable set of colour categories for the map's species legend rather than the 100+ distinct genera actually present in the data.

In [31]:
map_trees = Trees[['com_id', 'common_name', 'scientific_name', 'genus', 
                     'precinct', 'located_in', 'latitude', 'longitude',
                     'useful_life_expectency']].copy()

# Colour category = genus (top 8 + Other)
top_genera = map_trees['genus'].value_counts().head(8).index.tolist()
print("Top 8 genera:", top_genera)

def assign_display_category(genus):
    return genus if genus in top_genera else 'Other'

map_trees['display_category'] = map_trees['genus'].apply(assign_display_category)
print(map_trees['display_category'].value_counts())

Top 8 genera: ['Eucalyptus', 'Acacia', 'Ulmus', 'Platanus', 'Corymbia', 'Allocasuarina', 'Casuarina', 'Quercus']
display_category
Other            30898
Eucalyptus       17687
Acacia            7667
Ulmus             6102
Platanus          5646
Corymbia          4694
Allocasuarina     4010
Casuarina         2770
Quercus           2584
Name: count, dtype: int64


### Merging in the Canopy Match Results

The `map_trees` is merged with `TREE_CANOPY_MATCH` on `com_id`, bringing in `polygon_id` and `match_type` from the spatial join done earlier. This is a left join, so every tree in `map_trees` is kept even if, in the rare case of the 15 no-match trees, no corresponding polygon was found. Those rows simply get a null polygon_id. Without this merge, `map_trees` would have no way of knowing which canopy polygon each tree belongs to.

In [32]:
map_trees = map_trees.merge(
    TREE_CANOPY_MATCH[['com_id', 'polygon_id', 'match_type']],
    on='com_id',
    how='left'
)

In [33]:
from sklearn.cluster import DBSCAN

In [34]:
coords = map_trees[['latitude', 'longitude']].values
db = DBSCAN(eps=0.0015, min_samples=4).fit(coords)
map_trees['cluster_id'] = db.labels_

print(f"Trees not in any cluster: {(map_trees['cluster_id'] == -1).sum()}")

# Build zone shapes
zones = []
for cluster_id, group in map_trees[map_trees['cluster_id'] != -1].groupby('cluster_id'):
    center_lat = group['latitude'].mean()
    center_lon = group['longitude'].mean()
    radius_deg = np.sqrt(
        (group['latitude'] - center_lat)**2 + (group['longitude'] - center_lon)**2
    ).max()
    zones.append({
        'cluster_id': cluster_id,
        'center_lat': center_lat,
        'center_lon': center_lon,
        'radius_m': radius_deg * 111000,
        'tree_count': len(group)
    })

zones_df = pd.DataFrame(zones)
print(zones_df.head())

Trees not in any cluster: 4
   cluster_id  center_lat  center_lon     radius_m  tree_count
0           0  -37.801892  144.950923  6516.081590       81738
1           1  -37.782453  144.910138   672.544678         104
2           2  -37.789056  144.903881   145.959924          82
3           3  -37.831081  144.914993   178.562643          28
4           4  -37.807369  144.916816   841.953878          78


In [35]:
no_match_trees = TREE_CANOPY_MATCH[TREE_CANOPY_MATCH['match_type'] == 'no_match']
print(no_match_trees)
# Cross-reference with Trees to see what they are
Trees[Trees['com_id'].isin(no_match_trees['com_id'])][['com_id','common_name','precinct','latitude','longitude']]

        com_id  polygon_id match_type  distance_m
9723   1895757         NaN   no_match         NaN
9724   1895758         NaN   no_match         NaN
9725   1895760         NaN   no_match         NaN
9747   1895439         NaN   no_match         NaN
9758   1895466         NaN   no_match         NaN
11295  1900730         NaN   no_match         NaN
13335  1898194         NaN   no_match         NaN
23378  1916618         NaN   no_match         NaN
30093  1898205         NaN   no_match         NaN
30094  1898206         NaN   no_match         NaN
48911  1895451         NaN   no_match         NaN
48918  1895467         NaN   no_match         NaN
48922  1895477         NaN   no_match         NaN
57399  1898207         NaN   no_match         NaN
63666  1895759         NaN   no_match         NaN


,com_id,common_name,precinct,latitude,longitude
9723,1895757,Australian teak,North Melbourne,-37.800737,144.940310
9724,1895758,Australian teak,North Melbourne,-37.800719,144.940154
9725,1895760,Australian teak,North Melbourne,-37.800705,144.940001
9747,1895439,Old man banksia,North Melbourne,-37.801490,144.940869
9758,1895466,Ivy curl,North Melbourne,-37.801451,144.940958
11295,1900730,London Plane,Melbourne,-37.833265,144.972319
13335,1898194,Flame Tree,North Melbourne,-37.801298,144.940784
23378,1916618,Australian teak,North Melbourne,-37.800728,144.940230
30093,1898205,Coastal Banksia,North Melbourne,-37.801469,144.940940
30094,1898206,Cape Chestnut,North Melbourne,-37.801462,144.940874


Of the full 82,064-tree spatial join, 15 trees (0.018%) returned no canopy match. Cross-referencing these against the trees dataset shows 13 of the 15 cluster tightly around a single location in North Melbourne, suggesting a small, genuine gap in the 2021 canopy mapping at that spot rather than a systematic issue with the matching logic.

### Merge in the canopy match data

I tested DBSCAN first, and it produced one cluster covering 60% of all trees while excluding 241 entirely, which isn't usable for local, walkable zones. The fixed 100m grid solved this by including every tree with an even, predictable zone size. DBSCAN stands for Density-Based Spatial Clustering of Applications with Noise. It's a common clustering algorithm, a method for automatically grouping data points together based on how close they are to each other in space.

Reference: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html

In [36]:
from sklearn.cluster import DBSCAN

# Try a much smaller radius — start with 0.0004 (~40m) as asked for an estimate from AI
db = DBSCAN(eps=0.0004, min_samples=5).fit(coords)
map_trees['cluster_id'] = db.labels_

print(f"Number of clusters: {map_trees['cluster_id'].nunique()}")
print(f"Trees not in any cluster: {(map_trees['cluster_id'] == -1).sum()}")

cluster_sizes = map_trees[map_trees['cluster_id'] != -1]['cluster_id'].value_counts()
print(cluster_sizes.describe())
print(cluster_sizes.head(10))

Number of clusters: 86
Trees not in any cluster: 241
count       85.000000
mean       962.552941
std       5449.521819
min          4.000000
25%         10.000000
50%         22.000000
75%         66.000000
max      49174.000000
Name: count, dtype: float64
cluster_id
0     49174
12     8151
28     7851
23     2950
32     2603
24     2062
38     1619
11     1252
40     1082
25      867
Name: count, dtype: int64


Rather than clustering trees based on their proximity to each other, this method divides the entire study area into a fixed grid of 100m × 100m cells (grid_size = 0.001 degrees, approximating 100m at Melbourne's latitude). Each tree's coordinates are rounded to the nearest grid cell, and a cluster_id is generated by combining its rounded latitude and longitude. Because the grid exists uniformly across the whole map, every tree is guaranteed to fall inside a cell, unlike density-based clustering, no trees are excluded as noise, and no single cell can grow disproportionately large. This produced 2,388 grid-based zones with a consistent, predictable size, a more reliable foundation for the walkable "zones" the app needed than the highly uneven clusters DBSCAN produced.

In [37]:
grid_size = 0.001  # roughly 100m at Melbourne's latitude, I adjusted so it looks better

map_trees['grid_lat'] = (map_trees['latitude'] / grid_size).round() * grid_size
map_trees['grid_lon'] = (map_trees['longitude'] / grid_size).round() * grid_size
map_trees['cluster_id'] = map_trees['grid_lat'].astype(str) + '_' + map_trees['grid_lon'].astype(str)

cell_sizes = map_trees['cluster_id'].value_counts()
print(cell_sizes.describe())
print(f"Number of grid cells: {map_trees['cluster_id'].nunique()}")

count    2388.000000
mean       34.362647
std        49.645792
min         1.000000
25%        10.000000
50%        20.000000
75%        37.000000
max       767.000000
Name: count, dtype: float64
Number of grid cells: 2388


The grid method produced 2,388 zones with a median of 20 trees and a maximum of 767, no trees excluded and no single zone dominating the dataset. I checked the largest resulting zone specifically, 767 trees, and confirmed all of them belonged to a single precinct, Parkville, a suburb known for dense tree cover, confirming this was a genuine feature of the data rather than a clustering artefact:

In [38]:
largest_cell = cell_sizes.idxmax()
print(f"Largest cell ID: {largest_cell}, size: {cell_sizes.max()}")
print(map_trees[map_trees['cluster_id'] == largest_cell]['precinct'].value_counts())

Largest cell ID: -37.795_144.952, size: 767
precinct
Parkville    767
Name: count, dtype: int64


The grid clustering assigned each tree a `cluster_id` based on its rounded coordinates, that rounding creates the grid cell boundaries, but it doesn't give you a clean, single centre point for the zone as a whole. This code turns "which cell does each tree belong to" into "here is one summary row per zone, with its real centre and tree count", so i want to create a `zones_df` table for the app's map, search, and density features all depend on.

In [39]:
# Rebuild zone summary with proper lat/lon centers (using mean, not first value, for accuracy)
zones = []
for cell_id, group in map_trees.groupby('cluster_id'):
    zones.append({
        'cluster_id': cell_id,
        'center_lat': group['latitude'].mean(),
        'center_lon': group['longitude'].mean(),
        'tree_count': len(group)
    })

zones_df = pd.DataFrame(zones)

Now each zone's centre coordinates are calculated as the mean latitude and longitude of all trees within it, rather than an arbitrary tree's position, so the zone's location on the map accurately reflects where its trees are actually concentrated.

In [40]:
# ============================================================
# 1. map_trees — the main per-tree table
# ============================================================
print("map_trees shape:", map_trees.shape)
print(map_trees.head())
print(map_trees.dtypes)
print(map_trees.isnull().sum())

# ============================================================
# 2. zones_df — the grid-based zone summary table
# ============================================================
print("\nzones_df shape:", zones_df.shape)
print(zones_df.head())
print(zones_df.dtypes)
print(zones_df['tree_count'].describe())

# ============================================================
# 3. TREE_CANOPY_MATCH — the spatial join table
# ============================================================
print("\nTREE_CANOPY_MATCH shape:", TREE_CANOPY_MATCH.shape)
print(TREE_CANOPY_MATCH.head())
print(TREE_CANOPY_MATCH['match_type'].value_counts())

# ============================================================
# 4. The final exported JSON — reload it to confirm it's correct
# ============================================================
import json
with open('touch_grass_map_data.json') as f:
    output_check = json.load(f)

print(f"\nExported zones: {len(output_check['zones'])}")
print(f"Exported trees: {len(output_check['trees'])}")

# head-equivalent for the JSON lists — just slice the first few records
print("\nFirst 5 zones:")
for z in output_check['zones'][:5]:
    print(z)

print("\nFirst 5 trees:")
for t in output_check['trees'][:5]:
    print(t)

map_trees shape: (82058, 15)
    com_id   common_name        scientific_name     genus precinct located_in  \
0  1023218  London Plane  Platanus x acerifolia  Platanus  Carlton     Street   
1  1023221  London Plane  Platanus x acerifolia  Platanus  Carlton     Street   
2  1023222  London Plane  Platanus x acerifolia  Platanus  Carlton     Street   
3  1023224  London Plane  Platanus x acerifolia  Platanus  Carlton     Street   
4  1023225  London Plane  Platanus x acerifolia  Platanus  Carlton     Street   

    latitude   longitude useful_life_expectency display_category  polygon_id  \
0 -37.804986  144.968114          31 - 40 years         Platanus     52957.0   
1 -37.805022  144.968432          31 - 40 years         Platanus     31718.0   
2 -37.805059  144.968739          21 - 30 years         Platanus     54364.0   
3 -37.805247  144.968378          31 - 40 years         Platanus     18753.0   
4 -37.805216  144.968078                Unknown         Platanus     57975.0   

  m

In [41]:
# Quick shape summary for all your tables at once
tables = {
    'map_trees': map_trees,
    'zones_df': zones_df,
    'TREE_CANOPY_MATCH': TREE_CANOPY_MATCH
}

for name, df in tables.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

# Confirm the merge worked correctly — every tree should have a match_type
print(map_trees['match_type'].value_counts(dropna=False))

# Confirm cluster_id coverage — every tree should belong to exactly one zone
print(f"Unique cluster_ids in map_trees: {map_trees['cluster_id'].nunique()}")
print(f"Unique cluster_ids in zones_df: {zones_df['cluster_id'].nunique()}")
# these two numbers should match — if not, something's inconsistent between the tables

map_trees: 82058 rows, 15 columns
zones_df: 2388 rows, 4 columns
TREE_CANOPY_MATCH: 82058 rows, 4 columns
match_type
exact       50358
nearby      31685
no_match       15
Name: count, dtype: int64
Unique cluster_ids in map_trees: 2388
Unique cluster_ids in zones_df: 2388


In [42]:
print(map_trees['cluster_id'].nunique())   # should be 2388
print(zones_df['cluster_id'].nunique())    # should be 2388
print(map_trees['cluster_id'].nunique() == zones_df['cluster_id'].nunique())  # should be True

2388
2388
True


In [43]:
map_trees.head()

,com_id,common_name,scientific_name,genus,precinct,located_in,latitude,longitude,useful_life_expectency,display_category,polygon_id,match_type,cluster_id,grid_lat,grid_lon
0,1023218,London Plane,Platanus x acerifolia,Platanus,Carlton,Street,-37.804986,144.968114,31 - 40 years,Platanus,52957.0,exact,-37.805_144.968,-37.805,144.968
1,1023221,London Plane,Platanus x acerifolia,Platanus,Carlton,Street,-37.805022,144.968432,31 - 40 years,Platanus,31718.0,exact,-37.805_144.968,-37.805,144.968
2,1023222,London Plane,Platanus x acerifolia,Platanus,Carlton,Street,-37.805059,144.968739,21 - 30 years,Platanus,54364.0,exact,-37.805_144.969,-37.805,144.969
3,1023224,London Plane,Platanus x acerifolia,Platanus,Carlton,Street,-37.805247,144.968378,31 - 40 years,Platanus,18753.0,exact,-37.805_144.968,-37.805,144.968
4,1023225,London Plane,Platanus x acerifolia,Platanus,Carlton,Street,-37.805216,144.968078,Unknown,Platanus,57975.0,exact,-37.805_144.968,-37.805,144.968


In [44]:
zones_df.head()

,cluster_id,center_lat,center_lon,tree_count
0,-37.776_144.939,-37.775524,144.939279,5
1,-37.776_144.94,-37.775776,144.940104,57
2,-37.776_144.941,-37.775947,144.940983,55
3,-37.776_144.942,-37.775970,144.942037,43
4,-37.776_144.943,-37.776075,144.942854,41


In [45]:
Trees[Trees['year_planted'] == 1900]['age_description'].value_counts()

age_description
Mature         26512
Semi-mature        9
Name: count, dtype: int64

In [46]:
# You could even display this more informatively
def format_planted_year(row):
    if row['year_planted'] == 1900:
        return 'Established (exact year unknown)'
    return str(row['year_planted'])

map_trees['year_planted_display'] = Trees.apply(format_planted_year, axis=1)

Analysis of the 'year_planted' field revealed 26,521 records (32.3%) with a value of 1900, cross-referenced against age_description confirmed 99.97% of these are classified as 'Mature,' indicating 1900 was very likely used as a systematic placeholder for trees with an unrecorded planting date, rather than a genuine historical year. This was corrected to avoid presenting a fabricated exact year to users, while preserving the meaningful information that these trees are established/matur

In [47]:
# 1. Tree_canopy_2021 — raw canopy polygon dataset (with polygon_id added)
print("=== Tree_canopy_2021 ===")
print(Tree_canopy_2021.shape)
print(Tree_canopy_2021.head())

=== Tree_canopy_2021 ===
(57980, 3)
                                Geo Point  \
0  -37.795401772079884, 144.9578491713787   
1   -37.79535552368289, 144.9514339834316   
2  -37.79569214657717, 144.97057737168868   
3  -37.794822540469625, 144.9232541798998   
4   -37.79530213879436, 144.9476228687511   

                                           Geo Shape  polygon_id  
0  {"coordinates": [[[[144.95785054828673, -37.79...           0  
1  {"coordinates": [[[[144.95143147647892, -37.79...           1  
2  {"coordinates": [[[[144.9705639020907, -37.795...           2  
3  {"coordinates": [[[[144.9233177524972, -37.794...           3  
4  {"coordinates": [[[[144.94762531525055, -37.79...           4  


In [48]:
# 2. Trees — cleaned per-tree dataset
print("\n=== Trees ===")
Trees.head()


=== Trees ===


,com_id,common_name,scientific_name,genus,family,diameter_breast_height,year_planted,date_planted,age_description,useful_life_expectency,useful_life_expectency_value,precinct,located_in,coordinatelocation,latitude,longitude,easting,northing,geolocation
0,1023218,London Plane,Platanus x acerifolia,Platanus,Platanaceae,27.0,1900,1900-01-01,Mature,31 - 40 years,40,Carlton,Street,"-37.80498629, 144.968114",-37.804986,144.968114,321125.06,5813877.14,"-37.80498629, 144.968114"
1,1023221,London Plane,Platanus x acerifolia,Platanus,Platanaceae,35.0,1900,1900-01-01,Mature,31 - 40 years,40,Carlton,Street,"-37.80502242, 144.96843217",-37.805022,144.968432,321153.16,5813873.74,"-37.80502242, 144.96843217"
2,1023222,London Plane,Platanus x acerifolia,Platanus,Platanaceae,45.0,1900,1900-01-01,Mature,21 - 30 years,30,Carlton,Street,"-37.80505925, 144.96873896",-37.805059,144.968739,321180.25,5813870.24,"-37.80505925, 144.96873896"
3,1023224,London Plane,Platanus x acerifolia,Platanus,Platanaceae,NaN,1900,1900-01-01,Mature,31 - 40 years,40,Carlton,Street,"-37.80524724, 144.9683783",-37.805247,144.968378,321148.95,5813848.69,"-37.80524724, 144.9683783"
4,1023225,London Plane,Platanus x acerifolia,Platanus,Platanaceae,NaN,1900,1900-01-01,Mature,Unknown,50,Carlton,Street,"-37.80521592, 144.96807817",-37.805216,144.968078,321122.45,5813851.59,"-37.80521592, 144.96807817"


In [49]:
print("\n=== TREE_CANOPY_MATCH ===")
TREE_CANOPY_MATCH.head()


=== TREE_CANOPY_MATCH ===


,com_id,polygon_id,match_type,distance_m
0,1023218,52957.0,exact,0.0
1,1023221,31718.0,exact,0.0
2,1023222,54364.0,exact,0.0
3,1023224,18753.0,exact,0.0
4,1023225,57975.0,exact,0.0


In [50]:
print("\n=== map_trees ===")
map_trees.head()


=== map_trees ===


,com_id,common_name,scientific_name,genus,precinct,located_in,latitude,longitude,useful_life_expectency,display_category,polygon_id,match_type,cluster_id,grid_lat,grid_lon,year_planted_display
0,1023218,London Plane,Platanus x acerifolia,Platanus,Carlton,Street,-37.804986,144.968114,31 - 40 years,Platanus,52957.0,exact,-37.805_144.968,-37.805,144.968,Established (exact year unknown)
1,1023221,London Plane,Platanus x acerifolia,Platanus,Carlton,Street,-37.805022,144.968432,31 - 40 years,Platanus,31718.0,exact,-37.805_144.968,-37.805,144.968,Established (exact year unknown)
2,1023222,London Plane,Platanus x acerifolia,Platanus,Carlton,Street,-37.805059,144.968739,21 - 30 years,Platanus,54364.0,exact,-37.805_144.969,-37.805,144.969,Established (exact year unknown)
3,1023224,London Plane,Platanus x acerifolia,Platanus,Carlton,Street,-37.805247,144.968378,31 - 40 years,Platanus,18753.0,exact,-37.805_144.968,-37.805,144.968,Established (exact year unknown)
4,1023225,London Plane,Platanus x acerifolia,Platanus,Carlton,Street,-37.805216,144.968078,Unknown,Platanus,57975.0,exact,-37.805_144.968,-37.805,144.968,Established (exact year unknown)


In [51]:
print("\n=== zones_df ===")
zones_df.head()


=== zones_df ===


,cluster_id,center_lat,center_lon,tree_count
0,-37.776_144.939,-37.775524,144.939279,5
1,-37.776_144.94,-37.775776,144.940104,57
2,-37.776_144.941,-37.775947,144.940983,55
3,-37.776_144.942,-37.775970,144.942037,43
4,-37.776_144.943,-37.776075,144.942854,41


In [52]:
# ---- Assign each zone a precinct label ----
# A zone is labelled with whichever precinct the majority of its trees belong to.
# Ties are broken alphabetically (first precinct name wins).

def get_majority_precinct(group):
    counts = group['precinct'].value_counts()
    max_count = counts.max()
    tied = sorted(counts[counts == max_count].index)
    return tied[0]  # alphabetically first

zone_precincts = (
    map_trees
    .groupby('cluster_id')
    .apply(get_majority_precinct)
    .reset_index()
)
zone_precincts.columns = ['cluster_id', 'precinct']

# Merge the precinct label onto zones_df
zones_df = zones_df.merge(zone_precincts, on='cluster_id', how='left')

# Sanity checks
print(zones_df.shape)
print(zones_df['precinct'].isnull().sum())  # should be 0
print(zones_df['precinct'].value_counts().head(10))

(2388, 5)
0
precinct
Melbourne          456
Parkville          329
Kensington         209
North Melbourne    204
Docklands          192
West Melbourne     190
Carlton            185
East Melbourne     169
Southbank          123
South Yarra        117
Name: count, dtype: int64


/var/folders/g9/k5frvw7n1q79v7m2mp9fhv180000gn/T/ipykernel_12969/290847387.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_majority_precinct)


In [53]:
zone_precincts = (
    map_trees
    .groupby('cluster_id')
    .apply(get_majority_precinct, include_groups=False)
    .reset_index()
)

In [54]:
zones_df

,cluster_id,center_lat,center_lon,tree_count,precinct
0,-37.776_144.939,-37.775524,144.939279,5,Parkville
1,-37.776_144.94,-37.775776,144.940104,57,Parkville
2,-37.776_144.941,-37.775947,144.940983,55,Parkville
3,-37.776_144.942,-37.775970,144.942037,43,Parkville
4,-37.776_144.943,-37.776075,144.942854,41,Parkville
...,...,...,...,...,...
2383,-37.851_144.984,-37.850518,144.983756,2,Melbourne
2384,-37.85_144.981,-37.849798,144.980798,10,Melbourne
2385,-37.85_144.982,-37.850308,144.982030,9,Melbourne
2386,-37.85_144.983,-37.850426,144.982969,8,Melbourne


In [55]:
map_trees = Trees[['com_id', 'common_name', 'scientific_name', 'genus', 'family',
                     'precinct', 'located_in', 'latitude', 'longitude',
                     'useful_life_expectency']].copy()

map_trees['year_planted_display'] = Trees['year_planted'].apply(
    lambda y: 'Established (exact year unknown)' if y == 1900 else str(y)
)

top_genera = map_trees['genus'].value_counts().head(8).index.tolist()
print("Top 8 genera:", top_genera)

map_trees['display_category'] = map_trees['genus'].apply(
    lambda g: g if g in top_genera else 'Other'
)

# NEW ---- Family category (top 8 + Other) — used by the Tree Family filter ----
top_families = map_trees['family'].value_counts().head(8).index.tolist()
print("Top 8 families:", top_families)

map_trees['family_display_category'] = map_trees['family'].apply(
    lambda f: f if f in top_families else 'Other'
)
# NEW ---- end ----

map_trees = map_trees.merge(
    TREE_CANOPY_MATCH[['com_id', 'polygon_id', 'match_type']],
    on='com_id',
    how='left'
)

map_trees['grid_lat'] = (map_trees['latitude'] / grid_size).round() * grid_size
map_trees['grid_lon'] = (map_trees['longitude'] / grid_size).round() * grid_size
map_trees['cluster_id'] = map_trees['grid_lat'].astype(str) + '_' + map_trees['grid_lon'].astype(str)
map_trees = map_trees.drop(columns=['grid_lat', 'grid_lon'])

print(map_trees.shape)
print(map_trees[['cluster_id', 'display_category', 'polygon_id', 'match_type']].isnull().sum())
print("Unique zones in map_trees:", map_trees['cluster_id'].nunique())
print("Unique zones in zones_df:", zones_df['cluster_id'].nunique())
print("Missing family values:", map_trees['family'].isnull().sum())
print("Distinct families:", map_trees['family'].nunique())
print("zones_df has precinct column:", 'precinct' in zones_df.columns)
print("Missing precinct in zones_df:", zones_df['precinct'].isnull().sum() if 'precinct' in zones_df.columns else 'N/A')

# NEW ---- check the family bucketing worked well ----
print("family_display_category breakdown:")
print(map_trees['family_display_category'].value_counts())
# NEW ---- end ----

output = {
    'zones': zones_df.to_dict('records'),
    'trees': map_trees[['com_id', 'common_name', 'scientific_name', 'family',
                          'family_display_category', 'latitude', 'longitude',
                          'display_category', 'useful_life_expectency',
                          'year_planted_display', 'polygon_id', 'match_type',
                          'cluster_id', 'precinct', 'located_in']].to_dict('records')
}

with open('touch_grass_map_data.json', 'w') as f:
    json.dump(output, f)

print(f"Exported {len(zones_df)} zones and {len(map_trees)} trees")

Top 8 genera: ['Eucalyptus', 'Acacia', 'Ulmus', 'Platanus', 'Corymbia', 'Allocasuarina', 'Casuarina', 'Quercus']
Top 8 families: ['Myrtaceae', 'Fabaceae', 'Ulmaceae', 'Casuarinaceae', 'Platanaceae', 'Fagaceae', 'Proteaceae', 'Sapindaceae']
(82058, 16)
cluster_id           0
display_category     0
polygon_id          15
match_type           0
dtype: int64
Unique zones in map_trees: 2388
Unique zones in zones_df: 2388
Missing family values: 0
Distinct families: 64
zones_df has precinct column: True
Missing precinct in zones_df: 0
family_display_category breakdown:
family_display_category
Myrtaceae        29523
Other            17976
Fabaceae          8303
Ulmaceae          6820
Casuarinaceae     6780
Platanaceae       5646
Fagaceae          2590
Proteaceae        2491
Sapindaceae       1929
Name: count, dtype: int64
Exported 2388 zones and 82058 trees


In [56]:
map_trees['family'].value_counts()

family
Myrtaceae        29523
Fabaceae          8303
Ulmaceae          6820
Casuarinaceae     6780
Platanaceae       5646
                 ...  
Rubiaceae            1
Buxaceae             1
Taxaceae             1
Garryaceae           1
Theaceae             1
Name: count, Length: 64, dtype: int64

every time a user does something as simple as typing in the search box or tapping "Most Common," the frontend would have to loop through all 82,058 individual tree records just to figure out "how many distinct species are there, and how many of each." That's genuinely wasteful, since a user browsing Tree Explorer never needs to see individual trees, they need to see species, and there are only a few hundred distinct species total, not 82,058.

In [57]:
# ---- Build a lightweight species summary for Tree Explorer ----
# Groups all 82,058 individual trees down to one row per species,
# keeping only what the browsing/filtering UI actually needs.

species_summary = []

for sci_name, group in map_trees.groupby('scientific_name'):
    # which zones this species appears in, and how many times in each
    # (lets the frontend estimate "nearby" using zones_df's coordinates,
    #  instead of loading every individual tree's lat/lon)
    zone_counts = (
        group['cluster_id']
        .value_counts()
        .reset_index()
        .rename(columns={'count': 'count', 'cluster_id': 'cluster_id'})
        .to_dict('records')
    )

    species_summary.append({
        'scientific_name': sci_name,
        'common_name': group['common_name'].iloc[0],
        'family': group['family'].iloc[0],
        'family_display_category': group['family_display_category'].iloc[0],
        'display_category': group['display_category'].iloc[0],
        'total_count': len(group),
        'zone_counts': zone_counts,
    })

species_df = pd.DataFrame(species_summary)

print(species_df.shape)
print(species_df['total_count'].sum())
print(species_df.sort_values('total_count', ascending=False).head(5)[['common_name', 'total_count']])

output['species'] = species_df.to_dict('records')

with open('touch_grass_map_data.json', 'w') as f:
    json.dump(output, f)

print(f"Exported {len(zones_df)} zones, {len(map_trees)} trees, {len(species_df)} species")

(679, 7)
82058
         common_name  total_count
235    River red gum         8338
517     London Plane         5070
73   Drooping sheoak         3452
664      English Elm         3291
262       Yellow Box         3043
Exported 2388 zones, 82058 trees, 679 species


In [58]:
import json

with open('touch_grass_map_data.json') as f:
    data = json.load(f)

print("Top-level keys:", list(data.keys()))
print("Zones:", len(data['zones']))
print("Trees:", len(data['trees']))
print("Species:", len(data['species']))

Top-level keys: ['zones', 'trees', 'species']
Zones: 2388
Trees: 82058
Species: 679


In [59]:
print("Sample zone:", data['zones'][0])
print()
print("Sample tree:", data['trees'][0])
print()
print("Sample species:", data['species'][0])

Sample zone: {'cluster_id': '-37.776_144.939', 'center_lat': -37.7755242, 'center_lon': 144.93927874000002, 'tree_count': 5, 'precinct': 'Parkville'}

Sample tree: {'com_id': 1023218, 'common_name': 'London Plane', 'scientific_name': 'Platanus x acerifolia', 'family': 'Platanaceae', 'family_display_category': 'Platanaceae', 'latitude': -37.80498629, 'longitude': 144.968114, 'display_category': 'Platanus', 'useful_life_expectency': '31 - 40 years', 'year_planted_display': 'Established (exact year unknown)', 'polygon_id': 52957.0, 'match_type': 'exact', 'cluster_id': '-37.805_144.968', 'precinct': 'Carlton', 'located_in': 'Street'}

Sample species: {'scientific_name': 'Abies. sp', 'common_name': 'Fir', 'family': 'Pinaceae', 'family_display_category': 'Other', 'display_category': 'Other', 'total_count': 2, 'zone_counts': [{'cluster_id': '-37.832_144.912', 'count': 1}, {'cluster_id': '-37.835_144.91400000000002', 'count': 1}]}


In [60]:
print("Zone fields:", list(data['zones'][0].keys()))
print("Tree fields:", list(data['trees'][0].keys()))
print("Species fields:", list(data['species'][0].keys()))

Zone fields: ['cluster_id', 'center_lat', 'center_lon', 'tree_count', 'precinct']
Tree fields: ['com_id', 'common_name', 'scientific_name', 'family', 'family_display_category', 'latitude', 'longitude', 'display_category', 'useful_life_expectency', 'year_planted_display', 'polygon_id', 'match_type', 'cluster_id', 'precinct', 'located_in']
Species fields: ['scientific_name', 'common_name', 'family', 'family_display_category', 'display_category', 'total_count', 'zone_counts']


In [61]:
import pandas as pd

zones_check = pd.DataFrame(data['zones'])
trees_check = pd.DataFrame(data['trees'])
species_check = pd.DataFrame(data['species'])

zones_check.head()

,cluster_id,center_lat,center_lon,tree_count,precinct
0,-37.776_144.939,-37.775524,144.939279,5,Parkville
1,-37.776_144.94,-37.775776,144.940104,57,Parkville
2,-37.776_144.941,-37.775947,144.940983,55,Parkville
3,-37.776_144.942,-37.775970,144.942037,43,Parkville
4,-37.776_144.943,-37.776075,144.942854,41,Parkville


In [62]:
trees_check.head()

,com_id,common_name,scientific_name,family,family_display_category,latitude,longitude,display_category,useful_life_expectency,year_planted_display,polygon_id,match_type,cluster_id,precinct,located_in
0,1023218,London Plane,Platanus x acerifolia,Platanaceae,Platanaceae,-37.804986,144.968114,Platanus,31 - 40 years,Established (exact year unknown),52957.0,exact,-37.805_144.968,Carlton,Street
1,1023221,London Plane,Platanus x acerifolia,Platanaceae,Platanaceae,-37.805022,144.968432,Platanus,31 - 40 years,Established (exact year unknown),31718.0,exact,-37.805_144.968,Carlton,Street
2,1023222,London Plane,Platanus x acerifolia,Platanaceae,Platanaceae,-37.805059,144.968739,Platanus,21 - 30 years,Established (exact year unknown),54364.0,exact,-37.805_144.969,Carlton,Street
3,1023224,London Plane,Platanus x acerifolia,Platanaceae,Platanaceae,-37.805247,144.968378,Platanus,31 - 40 years,Established (exact year unknown),18753.0,exact,-37.805_144.968,Carlton,Street
4,1023225,London Plane,Platanus x acerifolia,Platanaceae,Platanaceae,-37.805216,144.968078,Platanus,Unknown,Established (exact year unknown),57975.0,exact,-37.805_144.968,Carlton,Street


In [63]:
species_check.head()

,scientific_name,common_name,family,family_display_category,display_category,total_count,zone_counts
0,Abies. sp,Fir,Pinaceae,Other,Other,2,"[{'cluster_id': '-37.832_144.912', 'count': 1}..."
1,Acacia acinacea,Gold-dust Wattle,Fabaceae,Fabaceae,Acacia,102,"[{'cluster_id': '-37.78_144.949', 'count': 36}..."
2,Acacia aneura,Mulga Wattle,Fabaceae,Fabaceae,Acacia,30,"[{'cluster_id': '-37.803_144.93', 'count': 11}..."
3,Acacia baileyana,Cootamundra Wattle,Fabaceae,Fabaceae,Acacia,16,"[{'cluster_id': '-37.782000000000004_144.949',..."
4,Acacia cultriformis,Knife-leaf Wattle,Fabaceae,Fabaceae,Acacia,3,"[{'cluster_id': '-37.792_144.958', 'count': 2}..."


In [64]:
print("Total trees across all species:", species_check['total_count'].sum())  # should be 82058
print("Any nulls in trees table:")
print(trees_check.isnull().sum())

Total trees across all species: 82058
Any nulls in trees table:
com_id                      0
common_name                 0
scientific_name             0
family                      0
family_display_category     0
latitude                    0
longitude                   0
display_category            0
useful_life_expectency      0
year_planted_display        0
polygon_id                 15
match_type                  0
cluster_id                  0
precinct                    0
located_in                  0
dtype: int64


In [65]:
print("Sum of zones_df tree_count:", zones_df['tree_count'].sum())
print("Total rows in map_trees:", len(map_trees))
print("These match:", zones_df['tree_count'].sum() == len(map_trees))

Sum of zones_df tree_count: 82058
Total rows in map_trees: 82058
These match: True
